# שיעור 09 - תבנית עיצוב מטה-קוגניציה


## הגדרה

פנקס רשימות זה מציג את תבנית העיצוב Metacognition באמצעות מסגרת הסוכן של מיקרוסופט.

**דרישות מוקדמות:**
- פריסת Azure OpenAI מוגדרת באמצעות משתני סביבה
- CLI של Azure מאומת (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## מהי מטאקוגניציה?

מטאקוגניציה היא **חשיבה על חשיבה**. בהקשר של סוכני AI, זה אומר לבנות סוכנים שיכולים:

- **להתבונן בעצמם** על התפוקות ותהליך ההסקה שלהם
- **לזהות טעויות** ולהתאושש בהדרגה במקום להיכשל בשקט
- **להעריך** האם התגובות שלהם שלמות ומועילות
- **להתאים** את האסטרטגיה שלהם כאשר גישה ראשונית לא עובדת (למשל, לחזור למערכת גיבוי)

סוכן מטאקוגניטיבי לא רק עונה על שאלות — הוא עוקב אחרי הביצועים שלו ומתאים את עצמו בזמן אמת.


## כלים ראשיים וגיבוי

דפוס מטאקוגניציה שכיח הוא **אסטרטגיית גיבוי**. הסוכן מנסה קודם כלי ראשי; אם הוא נכשל (למשל, שגיאת 404), הסוכן מזהה את הכישלון ומעביר באופן שקוף לכלי גיבוי.

זה משקף מערכות בעולם האמיתי שבהן שירותים ראשיים עשויים להיות לא זמינים והסוכנים חייבים לאבחן את הבעיה לפני בחירת דרך חלופית.

להלן נגדיר שני כלים לחיפוש טיסות:
- **ראשי** — מכסה את פריז, טוקיו וברצלונה
- **גיבוי** — מכסה את ברלין, סידני וניו יורק סיטי


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## סוכן המשתקף בעצמו עם התאוששות משגיאות

הסוכן שלמטה הונחה לנסות תחילה את מערכת הטיסה הראשית, לזהות כשלים, ולחזור לשקיפות למערכת הגיבוי. לאחר כל תגובה הוא מעריך בקצרה בעצמו אם השיב במלואו על שאלת המשתמש.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## תבנית הערכה עצמית

היבט נוסף של מטה-קוגניציה הוא **הערכה עצמית**: סוכן נפרד (או אותו סוכן בעבור שני) בוחן תגובה למלאות, דיוק, ולעזר.

למטה אנו יוצרים סוכן `ResponseEvaluator` שמדרג תגובות של סוכני נסיעות על שלושה מימדים.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## סיכום

בשיעור זה למדת כיצד לבנות **סוכנים מטאפגניים** באמצעות Microsoft Agent Framework:

- **השתקפות עצמית**: סוכנים שמנטרים את ההיגיון שלהם ומתקשרים בשקיפות את מה שקרה.
- **התאוששות מטעויות עם גיבויים**: דפוס של כלי ראשי + גיבוי בו הסוכן מזהה כשלים (למשל, שגיאות 404) ומנסה אוטומטית מקור חלופי.
- **הערכה עצמית**: סוכן מעריך נפרד שמדרג תגובות מבחינת שלימות, דיוק, ויעילות.

דפוסים אלו עושים את הסוכנים עמידים יותר, שקופים ואמינים — תכונות קריטיות לפריסות בייצור.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
